In [1]:
# ============================================================
# STRESS DETECTOR + LM STUDIO CHATBOT (FINAL CLEAN VERSION)
# ============================================================

import re
import pickle
import joblib
import requests
import nltk
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.sentiment.vader import SentimentIntensityAnalyzer



In [65]:
# ============================================================
# 1. LOAD NLTK VADER
# ============================================================
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

In [67]:
# ============================================================
# 2. LOAD MODEL + TOKENIZER
# ============================================================
print("Loading tokenizer...")
tokenizer = joblib.load("tokenizer.pkl")

print("Loading sequence config...")
config = joblib.load("sequence_config.pkl")
max_len = config["max_len"]

print("Loading model...")
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

print("All files loaded successfully!")

Loading tokenizer...
Loading sequence config...
Loading model...
All files loaded successfully!


In [69]:

# ============================================================
# 3. TEXT CLEANING
# ============================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [71]:
# ============================================================
# 4. STRESS PREDICTION FUNCTION
# ============================================================
def predict_stress(text):
    cleaned_text = clean_text(text)

    # LSTM Prediction
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=max_len, padding="post", truncating="post")

    prob = float(model.predict(padded, verbose=0)[0][0])  # 0 to 1

    # VADER Sentiment
    compound = sia.polarity_scores(cleaned_text)['compound']
    vader_scaled = (1 - compound) / 2  # convert to 0–1 stress-like

    # Hybrid Stress Score (Improved weighting)
    stress = (0.75 * prob) + (0.25 * vader_scaled)

    # Convert to 1–10 scale
    stress_level = round(1 + stress * 9, 2)
    stress_level = max(1.0, min(10.0, stress_level))

    # Probability (NOT true confidence)
    stress_percentage = round(prob * 100, 2)

    return stress_level, stress_percentage


In [73]:
# ============================================================
# 5. LM STUDIO API CALL (FIXED)
# ============================================================
def get_llm_response(prompt):
    url = "http://localhost:1234/v1/chat/completions"

    payload = {
        "model": "phi-3-mini",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.4,
        "max_tokens": 200
    }

    try:
        print("Thinking...")
        response = requests.post(url, json=payload, timeout=30)

        data = response.json()

        if "choices" not in data:
            return "Invalid response from LM Studio"

        return data["choices"][0]["message"]["content"]

    except requests.exceptions.Timeout:
        return "Model is taking too long. Try again."

    except requests.exceptions.ConnectionError:
        return "Cannot connect to LM Studio. Is server running?"

    except Exception as e:
        return f"Error: {e}"

In [75]:
# ============================================================
# CHAT LOOP (FINAL IMPROVED VERSION)
# ============================================================

history_scores = []

print("\n💬 Chat started (type 'exit' to quit)\n")

while True:
    user_text = input("You: ")

    if user_text.lower() == "exit":
        print("Exiting chat...")
        break

    # =========================
    # 1. CURRENT STRESS
    # =========================
    stress_level, stress_percentage = predict_stress(user_text)

    # =========================
    # 2. STORE HISTORY
    # =========================
    history_scores.append(stress_level)
    recent_scores = history_scores[-5:]
    avg_stress = sum(recent_scores) / len(recent_scores)

    # =========================
    # 3. PRINT INFO (CLEAN FORMAT)
    # =========================
    print("\n--- Analysis ---")
    print(f"Current Stress Level : {stress_level}/10")
    print(f"Model Score          : {stress_percentage}%")
    print(f"Average Stress (5)   : {round(avg_stress,2)}/10")

    # =========================
    # 4. SMART DECISION LOGIC
    # =========================
    if stress_level >= 7:
        tone = "high"
    elif avg_stress >= 6:
        tone = "medium"
    else:
        tone = "low"

    # =========================
    # 5. PROMPT BASED ON TONE
    # =========================
    if tone == "high":
        prompt = f"""
You are a supportive mental health assistant.

User message: "{user_text}"
Current stress level: {stress_level}/10

Give a calm, comforting response.
"""
    elif tone == "medium":
        prompt = f"""
You are a slightly empathetic assistant.

User message: "{user_text}"

Respond kindly and casually.
"""
    else:
        prompt = f"""
You are a friendly assistant.

User message: "{user_text}"

Reply normally.
"""

    # =========================
    # 6. GET RESPONSE
    # =========================
    print("Generating response...")
    reply = get_llm_response(prompt)

    print("\nBot:", reply)


💬 Chat started (type 'exit' to quit)



You:  Hello



--- Analysis ---
Current Stress Level : 2.37/10
Model Score          : 3.61%
Average Stress (5)   : 2.37/10
Generating response...
Thinking...

Bot:  Hello! How can I assist you today?


You:  I am feeling happy



--- Analysis ---
Current Stress Level : 3.34/10
Model Score          : 28.61%
Average Stress (5)   : 2.85/10
Generating response...
Thinking...

Bot:  I'm glad to hear that you're feeling happy! It's always wonderful when we can share our positive emotions with others. Is there anything specific that made you feel this way today? I'd be delighted to know more about it if you'd like to share. :)


You:  exit


Exiting chat...
